# Day 3 — The agent loop, budgets and planning
**Theory:** [day-03-agent-loop-and-planning.md](../theory/day-03-agent-loop-and-planning.md) · ⏱ ~75 min · Level: medium

**You will:**
1. Write a production-style agent loop from scratch (stop reasons, budgets, errors)
2. Add a `todo_write` planning tool and watch the plan evolve
3. Compare `effort` low vs high on cost, steps and quality
4. Add repeat detection

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("ANTHROPIC_API_KEY", UserSecretsClient().get_secret("ANTHROPIC_API_KEY"))
    except Exception as e:
        print("Add ANTHROPIC_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. Tools: reuse the ShopEase tools
They're scoped to one logged-in customer through a `Session`, as in production.

In [ ]:
import json, time
from shopease.tools import Session, TOOLS, dispatch

session = Session(customer_id="C001")
print([t["name"] for t in TOOLS])
print(dispatch("get_customer_overview", {}, session)[0][:300])

## 2. The loop, from scratch
Think → act → observe, plus three budgets (steps, $, seconds). Read every line. This is the core of every agent.

In [ ]:
def my_agent(user_msg, tools, run, system="", max_steps=12, max_cost=0.30, max_seconds=120, effort="medium"):
    messages, t0, cost0 = [{"role": "user", "content": user_msg}], time.time(), tracker.total()
    for step in range(1, max_steps + 1):
        r = client.messages.create(model=MODEL, max_tokens=8000, system=system, tools=tools,
                                   messages=messages, output_config={"effort": effort})
        tracker.add(MODEL, r.usage, f"step {step}")
        messages.append({"role": "assistant", "content": r.content})
        if r.stop_reason == "end_turn":
            return text_of(r), step, messages
        if r.stop_reason in ("max_tokens", "refusal"):
            return f"[stopped: {r.stop_reason}]", step, messages
        results = []
        for b in r.content:
            if b.type == "tool_use":
                out, err = run(b.name, b.input)
                print(f"  step {step} 🔧 {b.name}({json.dumps(b.input)[:70]}){' ❌' if err else ''}")
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": out, "is_error": err})
        messages.append({"role": "user", "content": results})
        if tracker.total() - cost0 > max_cost:
            return "[stopped: cost budget]", step, messages
        if time.time() - t0 > max_seconds:
            return "[stopped: time budget]", step, messages
    return "[stopped: step limit]", max_steps, messages

SYSTEM = "You are ShopEase support. Use tools before answering. Always check refund eligibility before promising refunds."
run = lambda name, args: dispatch(name, args, session)
answer, steps, transcript = my_agent("Can I get a refund for my coffee grinder?", TOOLS, run, SYSTEM)
print(f"\n({steps} steps)\n{answer}")

## 3. Planning with a todo list
A `todo_write` tool lets the model keep its plan near the *end* of the context, where attention is strongest.

In [ ]:
TODOS = []
def todo_write(items):
    TODOS[:] = items
    return "\n".join(f"[{ {'pending': ' ', 'in_progress': '~', 'done': 'x'}[i['status']] }] {i['task']}" for i in items)

TODO_TOOL = {"name": "todo_write", "description": "Write/replace your task list. Use it for multi-step tasks; "
             "update statuses as you go.",
             "input_schema": {"type": "object", "properties": {"items": {"type": "array", "items": {
                 "type": "object", "properties": {"task": {"type": "string"},
                                                  "status": {"type": "string", "enum": ["pending", "in_progress", "done"]}},
                 "required": ["task", "status"]}}}, "required": ["items"]}}

def run_with_todo(name, args):
    if name == "todo_write":
        out = todo_write(**args); print("   📝\n" + "\n".join("      " + l for l in out.splitlines()))
        return out, False
    return dispatch(name, args, session)

task = ("For each of my orders that is delivered, tell me if it can still be returned, "
        "then summarise everything in a short table.")
answer, steps, _ = my_agent(task, TOOLS + [TODO_TOOL], run_with_todo, SYSTEM)
print(f"\n({steps} steps)\n{answer}")

## 4. Effort: low vs high
Same task, two effort levels. Effort controls how much the model thinks (and how many tokens it spends).

In [ ]:
import pandas as pd
rows = []
for effort in ("low", "high"):
    before, t0 = tracker.total(), time.time()
    ans, steps, _ = my_agent("Which of my orders can I still return, and why?", TOOLS, run, SYSTEM, effort=effort)
    rows.append({"effort": effort, "steps": steps, "seconds": round(time.time() - t0, 1),
                 "cost $": round(tracker.total() - before, 4), "answer": ans[:120]})
pd.DataFrame(rows)

## 5. Repeat detection
A common failure: the same tool call with the same input, again and again. Detect it and tell the model.

In [ ]:
from collections import Counter
seen = Counter()
def run_guarded(name, args):
    key = (name, json.dumps(args, sort_keys=True))
    seen[key] += 1
    if seen[key] >= 3:
        return f"You already called {name} with this input {seen[key] - 1} times. Use the earlier result.", True
    return dispatch(name, args, session)

# simulate a stuck agent without calling the API
for _ in range(3):
    print(run_guarded("get_order", {"order_id": "O2001"})[0][:80])

## 6. Compare with the course helper
`course_utils.run_agent` is the same loop, packaged. You'll use it from Week 2 on.

In [ ]:
from course_utils import run_agent
fns = {t["name"]: (lambda n: lambda **a: dispatch(n, a, session)[0])(t["name"]) for t in TOOLS}
res = run_agent([{"role": "user", "content": "Where is my smart watch?"}], TOOLS, fns, system=SYSTEM, effort="low")
print(res.stop_reason, res.steps, "steps,", len(res.tool_calls), "tool calls")

## ✏️ Exercises
1. Set `max_steps=2` and give a multi-step task. What does the customer see? How would you make that message better?
2. Implement **plan-and-execute**: ask Opus for a numbered plan (structured output), then run each step with Haiku. Compare cost with ReAct.
3. Add a 4th budget: max tool calls per tool name.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker`. Make a habit of checking cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None